# 05 · Entrenamiento final y exportación de artefactos

## Objetivo

Los notebooks anteriores ya resolvieron:

1. **Dónde existe oportunidad de negocio**.
2. **Qué segmentos priorizar**.
3. **Cómo construir el panel temporal**.
4. **Qué modelos funcionan mejor y cómo validarlos**.

Este notebook tiene un propósito diferente:

> **Entrenar la versión final aprobada del forecast y generar artefactos reutilizables fuera de Colab.**

No se realizan nuevos experimentos ni selección de algoritmos.

### Arquitectura aprobada

- `H1` → **CatBoost MAE**
- `H2` → **CatBoost MAE**
- `H3` → **CatBoost MAE**
- `H4` → **Media móvil de 8 semanas**

La salida de este notebook será la base para la posterior contenerización del modelo.

## Artefactos esperados

La ejecución generará una estructura similar a:

```text
artifacts/
├── model_h1.cbm
├── model_h2.cbm
├── model_h3.cbm
├── model_config.json
├── feature_schema.json
├── metadata.json
├── buffers_demanda.csv
└── forecast_reference.csv
```

`H4` no requiere un archivo de modelo porque utiliza una regla determinista (`MEDIA_MOVIL_8`).

## 0. Archivos requeridos

- `panel_cluster4_v2.csv`
- `calendario_venezuela_ml_2025_2027.xlsx`
- `config_modelo_demanda.json`
- `buffers_demanda.csv`

Los dos últimos son generados por el notebook 04.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import importlib.util
import subprocess
import sys

import pandas as pd
import numpy as np

if importlib.util.find_spec("catboost") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "catboost"
    ])

import catboost
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def buscar_archivo(nombre):
    candidatos = [
        Path("/content") / nombre,
        Path("/mnt/data") / nombre,
        Path(nombre)
    ]
    return next((p for p in candidatos if p.exists()), None)

PANEL_PATH = buscar_archivo("panel_cluster4_v2.csv")
CALENDAR_PATH = buscar_archivo("calendario_venezuela_ml_2025_2027.xlsx")
CONFIG_PATH = buscar_archivo("config_modelo_demanda.json")
BUFFERS_PATH = buscar_archivo("buffers_demanda.csv")

archivos = {
    "Panel": PANEL_PATH,
    "Calendario": CALENDAR_PATH,
    "Configuración": CONFIG_PATH,
    "Buffers": BUFFERS_PATH
}

for nombre, ruta in archivos.items():
    print(f"{nombre:15}: {ruta}")

faltantes = [nombre for nombre, ruta in archivos.items() if ruta is None]

if faltantes:
    raise FileNotFoundError(
        "Faltan archivos: "
        + ", ".join(faltantes)
        + ". Ejecuta primero los notebooks 03 y 04."
    )

ARTIFACT_DIR = Path("/mnt/data/artifacts_modelo_demanda")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("\nDirectorio de artefactos:", ARTIFACT_DIR)

## 1. Carga de configuración aprobada

In [ ]:
with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    config_validada = json.load(f)

buffers_horizonte = pd.read_csv(BUFFERS_PATH)

display(pd.DataFrame(config_validada["horizontes"]).T)
display(buffers_horizonte)

### Control

La configuración productiva debe coincidir con la decisión metodológica del notebook 04. Se valida explícitamente antes de entrenar.

In [ ]:
seleccion_esperada = {
    "H1": "CB_MAE",
    "H2": "CB_MAE",
    "H3": "CB_MAE",
    "H4": "MEDIA_8"
}

for horizonte, modelo in seleccion_esperada.items():
    encontrado = config_validada["horizontes"][horizonte]["modelo"]
    assert encontrado == modelo, (
        f"Configuración inesperada en {horizonte}: "
        f"{encontrado} != {modelo}"
    )

print("Configuración de horizontes validada.")

## 2. Carga del panel definitivo

In [ ]:
df_modelo = pd.read_csv(
    PANEL_PATH,
    parse_dates=["FECHA_SEMANA"]
)

df_modelo = (
    df_modelo
    .sort_values(["SERIE_ID", "FECHA_SEMANA"])
    .reset_index(drop=True)
)

print("Filas:", f"{len(df_modelo):,}")
print("Series:", df_modelo["SERIE_ID"].nunique())
print(
    "Periodo:",
    df_modelo["FECHA_SEMANA"].min().date(),
    "→",
    df_modelo["FECHA_SEMANA"].max().date()
)
print(
    "% ceros:",
    f"{df_modelo['N_SERVICIOS'].eq(0).mean():.2%}"
)

## 3. Funciones reutilizables de feature engineering

A partir de este punto se evita código específico de notebook. Las funciones de esta sección son candidatas directas a migrarse posteriormente a `src/features.py` dentro del contenedor.

La lógica debe ser idéntica en entrenamiento e inferencia.

In [ ]:
VARIABLES_CALENDARIO = [
    "N_FESTIVOS_TOTAL",
    "N_FESTIVOS_LV",
    "DIAS_LABORABLES_LV",
    "N_DIAS_VACACIONALES_REF",
    "PCT_DIAS_VACACIONALES_REF",
    "TIENE_FESTIVO",
    "TIENE_VACACIONAL_REF",
    "ES_SEMANA_SANTA",
    "ES_CARNAVAL",
    "ES_NAVIDAD_FIN_ANIO"
]

CATEGORICAS = [
    "SERIE_ID",
    "CODIGO_TRATAMIENTO",
    "CODIGO_MUNICIPIO"
]

FEATURES_BASE = [
    "SERIE_ID",
    "CODIGO_TRATAMIENTO",
    "CODIGO_MUNICIPIO",
    "LAG_0",
    "LAG_1",
    "LAG_2",
    "LAG_3",
    "LAG_4",
    "LAG_8",
    "MEDIA_MOVIL_4",
    "MEDIA_MOVIL_8",
    "MEDIA_MOVIL_12",
    "STD_MOVIL_4",
    "STD_MOVIL_8",
    "SEMANAS_ACTIVAS_4",
    "SEMANAS_ACTIVAS_8",
    "SEMANAS_DESDE_DEMANDA",
    "T"
]

def features_horizonte(h):
    return FEATURES_BASE + [
        f"MES_TARGET_H{h}",
        f"TRIMESTRE_TARGET_H{h}",
        f"SEMANA_ANIO_TARGET_H{h}",
        f"SEMANA_SIN_TARGET_H{h}",
        f"SEMANA_COS_TARGET_H{h}",
        f"N_FESTIVOS_TOTAL_H{h}",
        f"N_FESTIVOS_LV_H{h}",
        f"DIAS_LABORABLES_LV_H{h}",
        f"N_DIAS_VACACIONALES_REF_H{h}",
        f"PCT_DIAS_VACACIONALES_REF_H{h}",
        f"TIENE_FESTIVO_H{h}",
        f"TIENE_VACACIONAL_REF_H{h}",
        f"ES_SEMANA_SANTA_H{h}",
        f"ES_CARNAVAL_H{h}",
        f"ES_NAVIDAD_FIN_ANIO_H{h}"
    ]

def calcular_semanas_desde_demanda(x):
    resultado = []
    semanas = None

    for valor in x:
        if valor > 0:
            semanas = 0
        elif semanas is not None:
            semanas += 1

        resultado.append(semanas)

    return pd.Series(resultado, index=x.index)

In [ ]:
def construir_features_historicas(panel):
    df = (
        panel
        .sort_values(["SERIE_ID", "FECHA_SEMANA"])
        .copy()
    )

    fecha_min = df["FECHA_SEMANA"].min()

    df["T"] = (
        (
            df["FECHA_SEMANA"] - fecha_min
        ).dt.days / 7
    ).astype(int)

    df["LAG_0"] = df["N_SERVICIOS"]

    for lag in [1, 2, 3, 4, 8]:
        df[f"LAG_{lag}"] = (
            df
            .groupby("SERIE_ID")["N_SERVICIOS"]
            .shift(lag)
        )

    for ventana in [4, 8, 12]:
        df[f"MEDIA_MOVIL_{ventana}"] = (
            df
            .groupby("SERIE_ID")["N_SERVICIOS"]
            .transform(
                lambda x: x.rolling(
                    ventana,
                    min_periods=1
                ).mean()
            )
        )

    for ventana in [4, 8]:
        df[f"STD_MOVIL_{ventana}"] = (
            df
            .groupby("SERIE_ID")["N_SERVICIOS"]
            .transform(
                lambda x: x.rolling(
                    ventana,
                    min_periods=2
                ).std()
            )
        )

    for ventana in [4, 8]:
        df[f"SEMANAS_ACTIVAS_{ventana}"] = (
            df
            .groupby("SERIE_ID")["N_SERVICIOS"]
            .transform(
                lambda x: (x > 0)
                .rolling(
                    ventana,
                    min_periods=1
                )
                .sum()
            )
        )

    df["SEMANAS_DESDE_DEMANDA"] = (
        df
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .transform(calcular_semanas_desde_demanda)
    )

    return df

In [ ]:
def agregar_targets_y_calendario(df, calendario):
    salida = df.copy()

    cal_base = calendario.copy()
    cal_base["FECHA_SEMANA"] = pd.to_datetime(
        cal_base["FECHA_SEMANA"]
    )

    for h in [1, 2, 3, 4]:

        salida[f"TARGET_H{h}"] = (
            salida
            .groupby("SERIE_ID")["N_SERVICIOS"]
            .shift(-h)
        )

        salida[f"FECHA_TARGET_H{h}"] = (
            salida["FECHA_SEMANA"]
            + pd.to_timedelta(h, unit="W")
        )

        fecha = salida[f"FECHA_TARGET_H{h}"]

        salida[f"MES_TARGET_H{h}"] = fecha.dt.month
        salida[f"TRIMESTRE_TARGET_H{h}"] = fecha.dt.quarter

        semana = (
            fecha
            .dt.isocalendar()
            .week
            .astype(float)
        )

        salida[f"SEMANA_ANIO_TARGET_H{h}"] = semana

        salida[f"SEMANA_SIN_TARGET_H{h}"] = (
            np.sin(2 * np.pi * semana / 52)
        )

        salida[f"SEMANA_COS_TARGET_H{h}"] = (
            np.cos(2 * np.pi * semana / 52)
        )

        cal = cal_base[
            ["FECHA_SEMANA"] + VARIABLES_CALENDARIO
        ].copy()

        renombres = {
            "FECHA_SEMANA": f"FECHA_TARGET_H{h}"
        }

        renombres.update({
            v: f"{v}_H{h}"
            for v in VARIABLES_CALENDARIO
        })

        cal = cal.rename(columns=renombres)

        salida = salida.merge(
            cal,
            on=f"FECHA_TARGET_H{h}",
            how="left"
        )

    return salida

## 4. Reconstrucción reproducible del dataset de entrenamiento

In [ ]:
calendario = pd.read_excel(
    CALENDAR_PATH,
    sheet_name="Calendario_Semanal"
)

df_features = construir_features_historicas(
    df_modelo
)

df_features = agregar_targets_y_calendario(
    df_features,
    calendario
)

features_historicas = [
    "LAG_0",
    "LAG_1",
    "LAG_2",
    "LAG_3",
    "LAG_4",
    "LAG_8",
    "MEDIA_MOVIL_4",
    "MEDIA_MOVIL_8",
    "MEDIA_MOVIL_12",
    "STD_MOVIL_4",
    "STD_MOVIL_8",
    "SEMANAS_ACTIVAS_4",
    "SEMANAS_ACTIVAS_8",
    "SEMANAS_DESDE_DEMANDA"
]

df_ml = (
    df_features
    .dropna(subset=features_historicas)
    .copy()
)

print("Filas ML:", f"{len(df_ml):,}")
print("Series ML:", df_ml["SERIE_ID"].nunique())

## 5. Fecha origen final

El entrenamiento final debe simular lo que realmente se conoce en la fecha de producción.

La fecha origen es la **última semana completa disponible** en el panel.

Para cada horizonte `h`, el entrenamiento solo puede utilizar filas cuyo target ya sea observable en esa fecha:

`FECHA_TARGET_Hh <= FECHA_ORIGEN_FINAL`

Esto evita leakage incluso durante el entrenamiento final.

In [ ]:
fecha_origen_final = df_ml["FECHA_SEMANA"].max()

forecast_base = df_ml[
    df_ml["FECHA_SEMANA"] == fecha_origen_final
].copy()

print("Fecha origen final:", fecha_origen_final.date())
print(
    "Series para forecast:",
    forecast_base["SERIE_ID"].nunique()
)

display(
    forecast_base[
        [
            "SERIE_ID",
            "DESCPROCED",
            "DESCMUNICIPIO",
            "N_SERVICIOS",
            "MEDIA_MOVIL_8"
        ]
    ]
    .sort_values("MEDIA_MOVIL_8", ascending=False)
    .head(10)
)

### Importante

Solo se pronostican mercados presentes en la fecha origen final. Una serie que dejó de estar vigente no debe reaparecer únicamente porque exista en el histórico.

En producción, este filtro deberá reforzarse con **vigencia contractual/baremo actual**.

## 6. Hiperparámetros finales CatBoost

In [ ]:
PARAMS_CB_MAE = {
    "iterations": 600,
    "depth": 5,
    "learning_rate": 0.05,
    "loss_function": "MAE",
    "l2_leaf_reg": 5,
    "random_seed": 42,
    "verbose": False
}

display(
    pd.DataFrame(
        [PARAMS_CB_MAE],
        index=["CB_MAE"]
    ).T
)

## 7. Entrenamiento final H1–H3

In [ ]:
modelos = {}
resumen_entrenamiento = []

for h in [1, 2, 3]:

    target = f"TARGET_H{h}"
    fecha_target = f"FECHA_TARGET_H{h}"
    features = features_horizonte(h)

    train = df_ml[
        (df_ml[fecha_target] <= fecha_origen_final) &
        (df_ml[target].notna())
    ].copy()

    X_train = train[features].copy()
    y_train = train[target].copy()

    for col in CATEGORICAS:
        X_train[col] = X_train[col].astype(str)

    modelo = CatBoostRegressor(
        **PARAMS_CB_MAE
    )

    modelo.fit(
        X_train,
        y_train,
        cat_features=CATEGORICAS
    )

    modelos[f"H{h}"] = modelo

    resumen_entrenamiento.append({
        "HORIZONTE": f"H{h}",
        "MODELO": "CB_MAE",
        "FILAS_TRAIN": len(train),
        "SERIES_TRAIN": train["SERIE_ID"].nunique(),
        "FECHA_TARGET_MAX_TRAIN": train[fecha_target].max()
    })

resumen_entrenamiento = pd.DataFrame(
    resumen_entrenamiento
)

display(resumen_entrenamiento)

## 8. Serialización de modelos

In [ ]:
rutas_modelos = {}

for h in [1, 2, 3]:

    ruta = ARTIFACT_DIR / f"model_h{h}.cbm"

    modelos[f"H{h}"].save_model(
        str(ruta)
    )

    rutas_modelos[f"H{h}"] = ruta

    print(
        f"H{h}: {ruta.name} "
        f"({ruta.stat().st_size / 1024:.1f} KB)"
    )

## 9. Generación del forecast de referencia

Se genera una predicción inmediatamente después del entrenamiento. Esta salida servirá para validar que, al recargar los artefactos, se obtienen exactamente los mismos resultados.

Las predicciones se conservan como valores continuos: representan **demanda esperada**, no servicios enteros ya ocurridos.

In [ ]:
forecast_partes = []

for h in [1, 2, 3]:

    features = features_horizonte(h)

    X_pred = forecast_base[features].copy()

    for col in CATEGORICAS:
        X_pred[col] = X_pred[col].astype(str)

    pred = np.maximum(
        modelos[f"H{h}"].predict(X_pred),
        0
    )

    tmp = forecast_base[
        [
            "SERIE_ID",
            "CODIGO_TRATAMIENTO",
            "DESCPROCED",
            "CODIGO_MUNICIPIO",
            "DESCMUNICIPIO",
            "MEDIA_MOVIL_8"
        ]
    ].copy()

    tmp["HORIZONTE"] = f"H{h}"
    tmp["FECHA_ORIGEN"] = fecha_origen_final
    tmp["FECHA_PRONOSTICO"] = (
        fecha_origen_final
        + pd.to_timedelta(h, unit="W")
    )
    tmp["MODELO"] = "CB_MAE"
    tmp["PREDICCION"] = pred

    forecast_partes.append(tmp)

# H4 = Media 8.
tmp_h4 = forecast_base[
    [
        "SERIE_ID",
        "CODIGO_TRATAMIENTO",
        "DESCPROCED",
        "CODIGO_MUNICIPIO",
        "DESCMUNICIPIO",
        "MEDIA_MOVIL_8"
    ]
].copy()

tmp_h4["HORIZONTE"] = "H4"
tmp_h4["FECHA_ORIGEN"] = fecha_origen_final
tmp_h4["FECHA_PRONOSTICO"] = (
    fecha_origen_final
    + pd.to_timedelta(4, unit="W")
)
tmp_h4["MODELO"] = "MEDIA_8"
tmp_h4["PREDICCION"] = tmp_h4["MEDIA_MOVIL_8"]

forecast_partes.append(tmp_h4)

forecast_reference = (
    pd.concat(
        forecast_partes,
        ignore_index=True
    )
    .sort_values(
        ["HORIZONTE", "SERIE_ID"]
    )
    .reset_index(drop=True)
)

display(
    forecast_reference
    .groupby(
        [
            "HORIZONTE",
            "FECHA_PRONOSTICO",
            "MODELO"
        ]
    )
    .agg(
        DEMANDA_TOTAL=("PREDICCION", "sum"),
        DEMANDA_MEDIA=("PREDICCION", "mean"),
        MEDIANA=("PREDICCION", "median"),
        MAXIMO=("PREDICCION", "max"),
        MERCADOS=("SERIE_ID", "nunique")
    )
    .reset_index()
)

### Resultado de referencia de la PoC

En la ejecución original se obtuvieron aproximadamente:

| Horizonte | Demanda |
|---|---:|
| H1 | 56,29 |
| H2 | 59,53 |
| H3 | 75,50 |
| H4 | 98,88 |

Estos valores no se codifican como constantes. La tabla anterior debe calcularse siempre con los datos disponibles al momento de ejecutar el notebook.

## 10. Incorporación de escenarios BASE / P80 / P90

In [ ]:
forecast_reference = forecast_reference.merge(
    buffers_horizonte,
    on="HORIZONTE",
    how="left"
)

forecast_reference["ESCALA"] = np.maximum(
    forecast_reference["MEDIA_MOVIL_8"],
    1
)

forecast_reference["DEMANDA_BASE"] = (
    forecast_reference["PREDICCION"]
)

forecast_reference["DEMANDA_P80"] = (
    forecast_reference["DEMANDA_BASE"]
    + forecast_reference["BUFFER_P80"]
    * forecast_reference["ESCALA"]
)

forecast_reference["DEMANDA_P90"] = (
    forecast_reference["DEMANDA_BASE"]
    + forecast_reference["BUFFER_P90"]
    * forecast_reference["ESCALA"]
)

for col in [
    "DEMANDA_BASE",
    "DEMANDA_P80",
    "DEMANDA_P90"
]:
    forecast_reference[col] = np.maximum(
        forecast_reference[col],
        0
    )

display(
    forecast_reference
    .groupby(
        ["HORIZONTE", "FECHA_PRONOSTICO"]
    )
    .agg(
        BASE=("DEMANDA_BASE", "sum"),
        P80=("DEMANDA_P80", "sum"),
        P90=("DEMANDA_P90", "sum"),
        MERCADOS=("SERIE_ID", "nunique")
    )
    .reset_index()
)

## 11. Esquema de features para producción

Se guarda explícitamente el orden y tipo esperado de las variables. Esto evita un problema frecuente en producción: cargar correctamente el modelo pero construir las columnas en distinto orden o tipo.

In [ ]:
feature_schema = {
    "categorical_features": CATEGORICAS,
    "historical_features": FEATURES_BASE,
    "features_by_horizon": {
        f"H{h}": features_horizonte(h)
        for h in [1, 2, 3]
    },
    "h4_rule": {
        "model": "MEDIA_8",
        "source_column": "MEDIA_MOVIL_8"
    }
}

FEATURE_SCHEMA_PATH = (
    ARTIFACT_DIR / "feature_schema.json"
)

with open(
    FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_schema,
        f,
        ensure_ascii=False,
        indent=2
    )

print(FEATURE_SCHEMA_PATH)

## 12. Configuración del modelo productivo

In [ ]:
model_config = {
    "model_name": "forecast_demanda_proveedores_medicos",
    "model_version": "1.0.0",
    "granularity": "TRATAMIENTO_CIUDAD_SEMANA",
    "frequency": "W-MON",
    "forecast_horizons": {
        "H1": {
            "type": "catboost",
            "artifact": "model_h1.cbm"
        },
        "H2": {
            "type": "catboost",
            "artifact": "model_h2.cbm"
        },
        "H3": {
            "type": "catboost",
            "artifact": "model_h3.cbm"
        },
        "H4": {
            "type": "moving_average",
            "window": 8,
            "source_column": "MEDIA_MOVIL_8"
        }
    },
    "catboost_params": PARAMS_CB_MAE,
    "negative_prediction_policy": "clip_to_zero",
    "scenario_formula": (
        "BASE + BUFFER_QUANTILE * max(MEDIA_MOVIL_8, 1)"
    )
}

MODEL_CONFIG_PATH = (
    ARTIFACT_DIR / "model_config.json"
)

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print(MODEL_CONFIG_PATH)

## 13. Trazabilidad y metadatos

Para poder auditar el modelo se registra:

- versión;
- fecha de entrenamiento;
- periodo de datos;
- fecha origen;
- número de series;
- librería y versión;
- hiperparámetros;
- artefactos;
- hash del panel utilizado.

El hash ayuda a identificar exactamente qué versión del dataset produjo los modelos.

In [ ]:
def sha256_archivo(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloque)

    return h.hexdigest()

metadata = {
    "model_name": model_config["model_name"],
    "model_version": model_config["model_version"],
    "trained_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "data_start": str(
        df_modelo["FECHA_SEMANA"].min().date()
    ),
    "data_end": str(
        df_modelo["FECHA_SEMANA"].max().date()
    ),
    "forecast_origin": str(
        fecha_origen_final.date()
    ),
    "historical_series": int(
        df_modelo["SERIE_ID"].nunique()
    ),
    "forecast_series": int(
        forecast_base["SERIE_ID"].nunique()
    ),
    "panel_rows": int(len(df_modelo)),
    "panel_sha256": sha256_archivo(
        PANEL_PATH
    ),
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "catboost_version": catboost.__version__,
    "training_summary": (
        resumen_entrenamiento
        .assign(
            FECHA_TARGET_MAX_TRAIN=lambda x:
                x["FECHA_TARGET_MAX_TRAIN"]
                .astype(str)
        )
        .to_dict(orient="records")
    )
}

METADATA_PATH = ARTIFACT_DIR / "metadata.json"

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print(METADATA_PATH)

## 14. Guardar outputs complementarios

In [ ]:
BUFFERS_EXPORT_PATH = (
    ARTIFACT_DIR / "buffers_demanda.csv"
)

FORECAST_REFERENCE_PATH = (
    ARTIFACT_DIR / "forecast_reference.csv"
)

buffers_horizonte.to_csv(
    BUFFERS_EXPORT_PATH,
    index=False
)

forecast_reference.to_csv(
    FORECAST_REFERENCE_PATH,
    index=False
)

print(BUFFERS_EXPORT_PATH)
print(FORECAST_REFERENCE_PATH)

## 15. Smoke test: recargar los modelos desde disco

Guardar un modelo no es suficiente. Se comprueba que un proceso nuevo pueda:

1. cargar el `.cbm`;
2. reconstruir las features;
3. predecir;
4. obtener exactamente la misma salida.

Este test es especialmente importante antes de construir el contenedor.

In [ ]:
predicciones_recargadas = {}

for h in [1, 2, 3]:

    modelo_recargado = CatBoostRegressor()

    modelo_recargado.load_model(
        str(
            ARTIFACT_DIR / f"model_h{h}.cbm"
        )
    )

    features = features_horizonte(h)

    X_pred = forecast_base[features].copy()

    for col in CATEGORICAS:
        X_pred[col] = X_pred[col].astype(str)

    pred = np.maximum(
        modelo_recargado.predict(X_pred),
        0
    )

    predicciones_recargadas[f"H{h}"] = pred

    referencia = (
        forecast_reference[
            forecast_reference["HORIZONTE"]
            == f"H{h}"
        ]
        .sort_values("SERIE_ID")["PREDICCION"]
        .to_numpy()
    )

    # Orden equivalente al forecast de referencia.
    pred_ordenada = (
        pd.DataFrame({
            "SERIE_ID": forecast_base["SERIE_ID"],
            "PREDICCION": pred
        })
        .sort_values("SERIE_ID")["PREDICCION"]
        .to_numpy()
    )

    diferencia_max = np.max(
        np.abs(
            referencia - pred_ordenada
        )
    )

    print(
        f"H{h} diferencia máxima:",
        diferencia_max
    )

    assert np.allclose(
        referencia,
        pred_ordenada,
        atol=1e-10,
        rtol=1e-10
    )

print("\nSmoke test de modelos: OK")

## 16. Validación H4

In [ ]:
h4_recalculado = (
    forecast_base[
        [
            "SERIE_ID",
            "MEDIA_MOVIL_8"
        ]
    ]
    .rename(
        columns={
            "MEDIA_MOVIL_8": "H4_RECALCULADO"
        }
    )
)

h4_referencia = (
    forecast_reference[
        forecast_reference["HORIZONTE"] == "H4"
    ][
        ["SERIE_ID", "PREDICCION"]
    ]
)

check_h4 = h4_referencia.merge(
    h4_recalculado,
    on="SERIE_ID",
    how="inner"
)

diferencia_h4 = np.max(
    np.abs(
        check_h4["PREDICCION"]
        - check_h4["H4_RECALCULADO"]
    )
)

print(
    "Diferencia máxima H4:",
    diferencia_h4
)

assert np.allclose(
    check_h4["PREDICCION"],
    check_h4["H4_RECALCULADO"]
)

print("Regla H4 validada.")

## 17. Inventario final de artefactos

In [ ]:
inventario = []

for archivo in sorted(
    ARTIFACT_DIR.iterdir()
):
    if archivo.is_file():
        inventario.append({
            "ARTEFACTO": archivo.name,
            "TAMANO_KB": archivo.stat().st_size / 1024
        })

inventario = pd.DataFrame(inventario)

display(inventario)

# Conclusiones

Con este notebook queda cerrada la transición de experimentación a artefactos productivos:

1. H1–H3 se entrenan con `CatBoost MAE` utilizando todo el histórico que sería conocido en la fecha origen final.
2. H4 queda implementado como regla reproducible de media móvil de ocho semanas.
3. Los modelos CatBoost se serializan en formato nativo `.cbm`.
4. Se guarda el esquema exacto de features.
5. Se conserva la configuración del modelo y los buffers P80/P90.
6. Se genera metadata de trazabilidad, incluyendo hash del dataset.
7. Se crea un `forecast_reference.csv` para pruebas de regresión.
8. Se verifica que los modelos recargados producen las mismas predicciones que los modelos en memoria.

## Siguiente fase: contenerización

El siguiente paso ya no debería hacerse dentro de un notebook.

La lógica común debe extraerse a código Python, por ejemplo:

```text
src/
├── features.py
├── predictor.py
├── schemas.py
└── config.py

artifacts/
├── model_h1.cbm
├── model_h2.cbm
├── model_h3.cbm
├── model_config.json
├── feature_schema.json
├── metadata.json
└── buffers_demanda.csv
```

Después se podrá construir una interfaz de inferencia:

`datos históricos recientes + calendario → forecast H1-H4 → BASE/P80/P90`

y empaquetarla mediante Docker.

**El optimizador de proveedores se conectará posteriormente a esta salida de demanda; no forma parte del modelo de forecast.**